import cv2
import numpy as np

def bits_to_text(bits):
    """تبدیل رشته بیت‌ها به متن (UTF-8)"""
    # تقسیم رشته به گروه‌های ۸ بیتی
    byte_array = bytearray()
    for i in range(0, len(bits), 8):
        byte_bits = bits[i:i+8]
        if len(byte_bits) < 8:
            break
        byte = int(byte_bits, 2)
        byte_array.append(byte)
    try:
        text = byte_array.decode('utf-8')
    except UnicodeDecodeError:
        text = "[خطا در دیکد UTF-8]"
    return text

def extract_lsb(image_path, text_length_bits=None):
    """
    استخراج متن از تصویر با روش LSB.
    
    Args:
        image_path: مسیر تصویر حاوی پیام
        text_length_bits: طول پیام بر حسب بیت (اگر مشخص باشد)
    Returns:
        متن استخراج‌شده
    """
    # 1. بارگذاری تصویر
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"تصویر {image_path} یافت نشد.")
    
    # 2. استخراج LSB از تمام پیکسل‌ها
    height, width, channels = img.shape
    bits = []
    
    for row in range(height):
        for col in range(width):
            for ch in range(channels):
                # استخراج LSB: pixel_val & 1
                lsb = img[row, col, ch] & 1
                bits.append(str(lsb))
    
    bit_string = ''.join(bits)
    
    # 3. اگر طول پیام مشخص باشد، فقط همان تعداد بیت اول را بگیر
    if text_length_bits is not None:
        bit_string = bit_string[:text_length_bits]
    
    # 4. تبدیل بیت‌ها به متن
    extracted_text = bits_to_text(bit_string)
    
    return extracted_text, len(bit_string)

# مثال اجرا
if __name__ == "__main__":
    extracted_text, bit_length = extract_lsb(
        image_path="[/mnt/data/encoded_image.png"](https://storage.gapgpt.app/media/code_interpreter/d7bb0669-0d8a-4dfe-8ccb-d36d6feb06f0/encoded_image.png%22),
        text_length_bits=None  # اگر طول را نمی‌دانید None بگذارید
    )
    print(f"✅ استخراج موفق.")
    print(f"   متن استخراج‌شده:\n{extracted_text}")
    print(f"   طول رشته بیت‌های پردازش‌شده: {bit_length}")


In [3]:
# extract_lsb.py
import cv2

def decode_lsb(image_path: str) -> str:
    """
    Extract LSB Steganography (RGB):
    - 32 بیت اول: طول پیام به بایت
    - سپس همان تعداد بایت خوانده و UTF-8 decode می‌شود
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    h, w, _ = img.shape
    flat = img.reshape(-1, 3)

    bit_index = 0

    # Read 4 bytes length header
    length_bytes = bytearray()
    for _ in range(4):
        byte_val = 0
        for _ in range(8):
            pixel_idx = bit_index // 3
            channel = bit_index % 3
            bit = flat[pixel_idx, channel] & 1
            byte_val = (byte_val << 1) | bit
            bit_index += 1
        length_bytes.append(byte_val)

    length = int.from_bytes(length_bytes, byteorder="big")

    # Sanity check: length must fit capacity
    remaining_bits = h * w * 3 - 32
    if length * 8 > remaining_bits:
        raise ValueError("Corrupted/invalid length header (does not fit image capacity).")

    # Read payload
    data = bytearray()
    for _ in range(length):
        byte_val = 0
        for _ in range(8):
            pixel_idx = bit_index // 3
            channel = bit_index % 3
            bit = flat[pixel_idx, channel] & 1
            byte_val = (byte_val << 1) | bit
            bit_index += 1
        data.append(byte_val)

    return data.decode("utf-8")

if __name__ == "__main__":
    stego_image = "encoded_image.png"
    text = decode_lsb(stego_image)
    print(text)


زندگی ادامه دارد, حتی وقتی سخت است, حتی وقتی درد دارد, دردها مسیر را روشن می‌کنند, نه پایان را,  ما ادامه می‌دهیم, قوی‌تر, آگاه‌تر, زنده‌تر. با احترام علیرضا بهمنش راد  - @behmaneshrad
